# Data cleaning — `features_filtered_by_ai.csv`

Cleans the story-filtered feature table produced by `src/filtered_feature_generation.py`.

**Input:** `data/features_filtered_by_ai.csv`  
**Output:** overwrites `data/features_filtered_by_ai.csv`

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

DATA = Path("../data")
SRC = DATA / "features_filtered_by_ai.csv"
OUT = DATA / "features_filtered_by_ai.csv"

In [2]:
# Load
df = pd.read_csv(SRC)
print("Shape:", df.shape)
df.info()
df.head()

Shape: (364, 42)
<class 'pandas.DataFrame'>
RangeIndex: 364 entries, 0 to 363
Data columns (total 42 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   video_id              364 non-null    str    
 1   label                 364 non-null    str    
 2   n_words_calc          364 non-null    int64  
 3   lex_var_30            364 non-null    float64
 4   mattr_30              364 non-null    float64
 5   negation_count        364 non-null    int64  
 6   negation_rate         364 non-null    float64
 7   reaction_count        364 non-null    int64  
 8   reaction_rate         364 non-null    float64
 9   contraction_count     364 non-null    int64  
 10  contraction_rate      364 non-null    float64
 11  everything_count      364 non-null    int64  
 12  love_count            364 non-null    int64  
 13  well_count            364 non-null    int64  
 14  filler_count          364 non-null    int64  
 15  filler_rate      

,video_id,label,n_words_calc,lex_var_30,mattr_30,negation_count,negation_rate,reaction_count,reaction_rate,contraction_count,...,profanity_rate,digit_rate,first_person_rate,second_person_rate,we_rate,repeat_bigram_rate,repeat_sentence_rate,proper_noun_rate,cta_flag,y
0,5Y4lU1gtOzo,AI,200,0.9667,0.9228,4,2.0000,0,0.0000,2,...,0.0,0.0000,0.0000,0.0000,0.0000,2.5126,0.0000,0.0000,0,1
1,TSf7LOimamk,human,110,0.9000,0.8716,5,4.5455,1,0.9091,13,...,0.0,0.0000,12.7273,3.6364,0.0000,5.5046,18.1818,7.2727,0,0
2,Exemc-4bXPM,AI,242,0.8667,0.8136,17,7.0248,0,0.0000,24,...,0.0,0.4132,5.3719,8.2645,0.8264,11.2033,2.0833,0.8264,0,1
3,wI6Mc00aspk,AI,170,0.7333,0.8556,3,1.7647,0,0.0000,2,...,0.0,1.1765,11.1765,0.5882,0.0000,4.1420,0.0000,3.5294,0,1
4,MEGD2JP5yCA,AI,56,0.9000,0.9543,1,1.7857,0,0.0000,2,...,0.0,0.0000,1.7857,3.5714,0.0000,0.0000,0.0000,7.1429,1,1


In [3]:
# Keep modelling columns (drop free-text / id columns that leak the label)
drop_cols = [c for c in ["video_id", "label", "source_query", "title", "url", "channel",
                         "theme", "llm_category", "llm_confidence", "llm_reason"] if c in df.columns]
keep = [c for c in df.columns if c not in drop_cols or c in ("y", "n_words", "n_words_calc")]
# Always keep y and a length column if present
if "y" not in keep and "y" in df.columns:
    keep.append("y")
df = df[[c for c in df.columns if c in keep or c == "y"]].copy()

print("Columns kept:", list(df.columns))
print(df["y"].value_counts() if "y" in df.columns else "no y column")

Columns kept: ['n_words_calc', 'lex_var_30', 'mattr_30', 'negation_count', 'negation_rate', 'reaction_count', 'reaction_rate', 'contraction_count', 'contraction_rate', 'everything_count', 'love_count', 'well_count', 'filler_count', 'filler_rate', 'everything_rate', 'love_rate', 'well_rate', 'n_sentences', 'avg_sent_len', 'sent_len_std', 'max_sent_len', 'short_sent_rate', 'question_rate', 'exclaim_rate', 'ellipsis_rate', 'avg_word_len', 'long_word_rate', 'hapax_rate', 'ttr_full', 'top_word_share', 'profanity_rate', 'digit_rate', 'first_person_rate', 'second_person_rate', 'we_rate', 'repeat_bigram_rate', 'repeat_sentence_rate', 'proper_noun_rate', 'cta_flag', 'y']
y
0    192
1    172
Name: count, dtype: int64


In [4]:
# Basic quality filters
n_before = len(df)

# Drop rows with missing target
if "y" in df.columns:
    df = df.dropna(subset=["y"])

# Drop rows that are too short (prefer n_words_calc, fall back to n_words)
length_col = "n_words_calc" if "n_words_calc" in df.columns else ("n_words" if "n_words" in df.columns else None)
if length_col:
    df = df[df[length_col] > 5]

# Drop exact-duplicate feature rows (same numbers across all non-id columns)
feat_cols = [c for c in df.columns if c not in ("y", "video_id")]
df = df.drop_duplicates(subset=feat_cols)

print(f"Rows: {n_before} -> {len(df)}")
df.info()

Rows: 364 -> 364
<class 'pandas.DataFrame'>
RangeIndex: 364 entries, 0 to 363
Data columns (total 40 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   n_words_calc          364 non-null    int64  
 1   lex_var_30            364 non-null    float64
 2   mattr_30              364 non-null    float64
 3   negation_count        364 non-null    int64  
 4   negation_rate         364 non-null    float64
 5   reaction_count        364 non-null    int64  
 6   reaction_rate         364 non-null    float64
 7   contraction_count     364 non-null    int64  
 8   contraction_rate      364 non-null    float64
 9   everything_count      364 non-null    int64  
 10  love_count            364 non-null    int64  
 11  well_count            364 non-null    int64  
 12  filler_count          364 non-null    int64  
 13  filler_rate           364 non-null    float64
 14  everything_rate       364 non-null    float64
 15  love_rate        

In [5]:
# Sanity checks on the target and length
if "y" in df.columns:
    print(df["y"].value_counts())
if length_col:
    print(df[length_col].describe())

y
0    192
1    172
Name: count, dtype: int64
count    364.000000
mean     113.664835
std       62.497628
min       30.000000
25%       57.000000
50%      105.000000
75%      159.250000
max      281.000000
Name: n_words_calc, dtype: float64


In [ ]:
# Length distribution by label (if we still have a human-readable label)
plot_df = df.copy()
if "y" in plot_df.columns and "label" not in plot_df.columns:
    plot_df["label"] = plot_df["y"].map({0: "human", 1: "AI"})

if length_col and "label" in plot_df.columns:
    sns.histplot(data=plot_df, x=length_col, hue="label", kde=True,
                 stat="density", common_norm=False)
    plt.title("Distribution of Number of Words per Label")
    plt.xlabel("Number of words")
    plt.show()

    sns.boxplot(data=plot_df, x="label", y=length_col)
    plt.title("Word count by label")
    plt.show()

In [6]:
# Save cleaned table
df.to_csv(OUT, index=False)
print(f"Saved {len(df)} rows x {df.shape[1]} columns -> {OUT}")

Saved 364 rows x 40 columns -> ../data/features_filtered_by_ai.csv
